# Tutorial: Escala de Enfermagem com `Container` e `start_delayed()`

**Público:** alunos que já viram filas e agora querem modelar capacidade agregada ao longo do tempo.

**Pré-requisitos:** entender `timeout` e recursos básicos.

**Objetivos de aprendizagem:**

- usar `Container` para representar quantidade disponível;
- iniciar processos futuros com `start_delayed()`;
- interpretar sobreposição de turnos e consumo temporário de equipe.


## Roteiro

1. Entender por que este problema não usa `Resource`.
2. Modelar os turnos.
3. Modelar tarefas que consomem parte da equipe.
4. Rodar um dia completo.
5. Interpretar os momentos de folga e aperto operacional.


In [1]:
from __future__ import annotations

import simpy
from simpy.util import start_delayed

print("SimPy carregado com sucesso")

SimPy carregado com sucesso


## 1. Cenário

Queremos acompanhar a quantidade de profissionais disponíveis ao longo do dia.

Temos três turnos:

- manhã;
- tarde;
- noite.

E algumas tarefas que consomem parte da equipe por um período:

- medicação matinal;
- pico de admissões;
- medicação da noite.


## 2. Conceito fundamental: por que `Container`?

Aqui não importa **quem** é o enfermeiro.

Importa apenas **quantos profissionais estão disponíveis agora**.

Por isso `Container` é a escolha correta.

Se quiséssemos saber qual profissional está livre, sua skill mix ou sua escala individual, aí precisaríamos de outra modelagem.


In [2]:
BASE_HORA = 7


def relogio(t):
    h = int(BASE_HORA + t) % 24
    m = int(round((t - int(t)) * 60))
    return f"{h:02d}:{m:02d}"

### Pequena observação didática

A simulação usa tempo em horas a partir de `07:00`.

A função `relogio()` existe apenas para traduzir esse tempo simulado em uma saída mais amigável para aula.


In [3]:
def turno(env, equipe, nome, qtd, duracao):
    print(f"{relogio(env.now)} | entra turno {nome} (+{qtd})")
    yield equipe.put(qtd)
    print(f"{relogio(env.now)} | disponíveis={equipe.level:.0f}")
    yield env.timeout(duracao)
    yield equipe.get(qtd)
    print(
        f"{relogio(env.now)} | sai turno {nome} (-{qtd}) | disponíveis={equipe.level:.0f}"
    )


def tarefa(env, equipe, nome, inicio, duracao, qtd):
    yield env.timeout(inicio)
    yield equipe.get(qtd)
    print(
        f"{relogio(env.now)} | inicia {nome} (-{qtd}) | disponíveis={equipe.level:.0f}"
    )
    yield env.timeout(duracao)
    yield equipe.put(qtd)
    print(
        f"{relogio(env.now)} | termina {nome} (+{qtd}) | disponíveis={equipe.level:.0f}"
    )

## 3. Ideia central do notebook

Os turnos **colocam** capacidade no sistema.

As tarefas **retiram temporariamente** parte dessa capacidade.

Essa lógica é muito boa para discutir planejamento agregado sem ainda entrar em escala individual por pessoa.


In [4]:
def executar_simulacao():
    env = simpy.Environment()
    equipe = simpy.Container(env, capacity=20, init=0)

    env.process(turno(env, equipe, "MANHA", 6, 6.0))
    start_delayed(env, turno(env, equipe, "TARDE", 5, 6.5), 5.5)
    start_delayed(env, turno(env, equipe, "NOITE", 4, 12.5), 11.5)

    start_delayed(env, tarefa(env, equipe, "MEDICACAO_MATINAL", 0, 1.0, 2), 1.0)
    start_delayed(env, tarefa(env, equipe, "PICO_ADMISSOES", 0, 2.0, 3), 5.0)
    start_delayed(env, tarefa(env, equipe, "MEDICACAO_NOITE", 0, 1.0, 2), 13.0)

    env.run()


executar_simulacao()

07:00 | entra turno MANHA (+6)
07:00 | disponíveis=6
08:00 | inicia MEDICACAO_MATINAL (-2) | disponíveis=4
09:00 | termina MEDICACAO_MATINAL (+2) | disponíveis=6
12:00 | inicia PICO_ADMISSOES (-3) | disponíveis=3
12:30 | entra turno TARDE (+5)
12:30 | disponíveis=8
13:00 | sai turno MANHA (-6) | disponíveis=2
14:00 | termina PICO_ADMISSOES (+3) | disponíveis=5
18:30 | entra turno NOITE (+4)
18:30 | disponíveis=9
19:00 | sai turno TARDE (-5) | disponíveis=4
20:00 | inicia MEDICACAO_NOITE (-2) | disponíveis=2
21:00 | termina MEDICACAO_NOITE (+2) | disponíveis=4
07:00 | sai turno NOITE (-4) | disponíveis=0


## 4. O que observar na saída

Pontos principais:

- a passagem manhã -> tarde tem sobreposição e depois queda brusca;
- o pico de admissões consome parte importante da equipe;
- a entrada do turno da noite recompõe a disponibilidade.

Esse tipo de leitura é muito útil para discutir capacidade instalada e janelas de maior risco operacional.


## 5. Detalhe técnico importante

Neste notebook usamos `env.run()` até esgotar os eventos.

Isso evita o risco de cortar a última mensagem exatamente no marco final do horizonte, um detalhe que costuma confundir quem está começando em SimPy.


## 6. Exercícios

1. Aumente a tarefa `PICO_ADMISSOES` para consumir `4` profissionais.
2. Remova a sobreposição entre manhã e tarde. O que acontece?
3. Explique por que este problema ficaria mais pesado se fosse modelado com identidade individual de cada profissional.


In [5]:
# Espaço para experimentos:
# - ajuste quantidade e duração dos turnos;
# - ajuste a demanda das tarefas;
# - rode novamente e compare a disponibilidade ao longo do dia.

## 7. Extensão sugerida

Se você quiser aproximar isso da realidade, pode evoluir para:

- skill mix;
- pausas e descanso;
- jornada legal;
- enfermeiro por setor;
- profissional com identidade própria.

Nesse ponto, `Container` deixa de bastar sozinho.
